# Week 15, Day 3: The call stack, frame by frame

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w015d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def show(n):
    if n == 0:
        return
    show(n - 1)
    print(n)

show(3)

In [ ]:
def area(w, h):
    return w * h

def room_cost(w, h, price):
    return area(w, h) * price

def house_cost(rooms, price):
    return sum(room_cost(w, h, price) for w, h in rooms)

print(house_cost([(3, 4), (2, 5)], 1500))

> **Key idea.** A call pushes a frame holding that call's own variables; a return pops it. Each recursive call has its own frame, so its `n` is separate from every other call's `n`.

In [ ]:
def down_and_up(n):
    if n == 0:
        print("  base case")
        return
    print("going down", n)
    down_and_up(n - 1)
    print("coming up", n)

down_and_up(3)

*Interactive figure on the website: This version adds up 1 to n. Nothing can be added on the way down: each call waits for the smaller total, then adds its own n on the way back up..*

In [ ]:
def reverse(text):
    if not text:
        return ""
    return reverse(text[1:]) + text[0]

print(reverse("stressed"))

*Interactive figure on the website: The tree for fib(5). Look for repeats: how many times is fib(2) worked out? Then try fib(7)..*

In [ ]:
calls = 0

def fib(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

for n in [5, 10, 15, 20, 25]:
    calls = 0
    value = fib(n)
    print(f"fib({n}) = {value:>6}  took {calls:>7} calls")

In [ ]:
def digits_of(n, found=[]):
    found.insert(0, n % 10)
    if n >= 10:
        digits_of(n // 10, found)
    return found

print(digits_of(123))
print(digits_of(45))

In [ ]:
def digits_of(n):
    if n < 10:
        return [n]
    return digits_of(n // 10) + [n % 10]

print(digits_of(123), digits_of(45))

> **Tip.** Prefer recursive functions that **return** their results and build answers from the results of the smaller calls. They have no hidden state, so every call can be understood on its own: which is the whole point of the leap of faith.

### Exercise: Before or after?

Work out what each call prints, writing it as a string with `\n` between lines.

In [ ]:
# def a(n):
#     if n > 0:
#         print(n)
#         a(n - 1)
# a(3)
first = ___
# def b(n):
#     if n > 0:
#         b(n - 1)
#         print(n)
# b(3)
second = ___
# def c(n):
#     if n > 0:
#         print(n)
#         c(n - 1)
#         print(n)
# c(2)
third = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert first == \"3\\n2\\n1\"\n    assert second == \"1\\n2\\n3\"\n    assert third == \"2\\n1\\n1\\n2\"")

### Exercise: Up and down

Write `mountain(n)` that returns the list `[1, 2, ..., n, ..., 2, 1]` recursively: `mountain(3)` is `[1, 2, 3, 2, 1]`, `mountain(1)` is `[1]`. Hint: `mountain(n)` is `mountain(n - 1)` with something put in the middle... or think of it as `[1]` around a smaller mountain of bigger numbers. Either works; pick one and make the leap.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert mountain(3) == [1, 2, 3, 2, 1] and mountain(1) == [1] and mountain(5) == [1, 2, 3, 4, 5, 4, 3, 2, 1]\n    assert \"for \" not in source and \"while \" not in source and \"range\" not in source")

### Exercise: Ruler marks

The marks on a ruler between inch marks follow a recursive pattern: the middle mark is longest, and each half is a smaller ruler. Write `ruler(n)` that returns the pattern of mark lengths as a string: `ruler(1)` is `"1"`, `ruler(2)` is `"121"`, `ruler(3)` is `"1213121"`. Each `ruler(n)` is `ruler(n - 1)`, then `n`, then `ruler(n - 1)` again.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert ruler(1) == \"1\" and ruler(2) == \"121\" and ruler(3) == \"1213121\"\n    assert ruler(4) == \"121312141213121\" and len(ruler(9)) == 2 ** 9 - 1\n    assert ruler(0) == \"\"")

### Exercise: Counting the tree

Without running `fib` itself, write `fib_calls(n)` that returns how many calls `fib(n)` makes (including the first). The count follows the same shape as `fib`: a call to `fib(n)` is one call, plus all the calls made by `fib(n - 1)` and `fib(n - 2)`. Calls with `n < 2` make only themselves.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert [fib_calls(n) for n in range(8)] == [1, 1, 3, 5, 9, 15, 25, 41]\n    assert fib_calls(20) == 21891")

### Exercise: Show the calls

Write `trace_factorial(n)` that returns the lines a person tracing `factorial(n)` would write, indented two spaces per level, going down and coming back up:

```
trace_factorial(2)  ->
factorial(2)
  factorial(1)
    factorial(0)
    returns 1
  returns 1
returns 2
```

Return the lines as a list of strings. Give your function a `depth` parameter with a default of 0.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert trace_factorial(2) == [\"factorial(2)\", \"  factorial(1)\", \"    factorial(0)\", \"    returns 1\", \"  returns 1\", \"returns 2\"]\n    assert trace_factorial(0) == [\"factorial(0)\", \"returns 1\"]\n    assert len(trace_factorial(5)) == 12")

### Bug hunt: The list that remembers

`evens_down(n)` should return the even numbers from n down to 0. The first call is right, but later calls include numbers from earlier calls. Fix it so each call returns a fresh list.

In [ ]:
def evens_down(n, found=[]):
    if n < 0:
        return found
    if n % 2 == 0:
        found.append(n)
    return evens_down(n - 1, found)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"fresh every time\"\n    assert evens_down(6) == [6, 4, 2, 0]\n    assert evens_down(3) == [2, 0]\n    assert evens_down(-1) == []")

### Bug hunt: Backwards

`spell(word)` should return the letters of a word separated by dashes, in order: `"cat"` gives `"c-a-t"`. It gives `"t-a-c"`. Fix it by moving the work to the right side of the recursive call.

In [ ]:
def spell(word):
    if len(word) <= 1:
        return word
    return spell(word[1:]) + "-" + word[0]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert spell(\"cat\") == \"c-a-t\" and spell(\"a\") == \"a\" and spell(\"\") == \"\"")

### Puzzle: Print a list backwards

Arrange the lines so `print_backwards(items)` prints the items last first, one per line, recursively. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Stack questions

Work these out without running anything.

In [ ]:
a = ___   # the most frames of factorial on the stack at once during factorial(5)
b = ___   # the most frames of fib on the stack at once during fib(5)
c = ___   # the total number of calls fib(5) makes
d = ___   # reverse("abc") with the reverse function from today

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 6\n    assert b == 5\n    assert c == 15\n    assert d == \"cba\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the Sierpinski triangle

The **Sierpinski triangle**, described by Wacław Sierpiński in 1915, is a triangle made of three half-size copies of itself, each made of three smaller copies, and so on. Write `sierpinski(n)` that returns it as a list of strings using `*` and spaces, with `2 ** n` rows:

```
sierpinski(0):    sierpinski(1):    sierpinski(2):
*                  *                   *
                  * *                 * *
                                     *   *
                                    * * * *
```

The recursive idea: take the lines of `sierpinski(n - 1)`; the top half is each of those lines with spaces added on both sides to centre it; the bottom half is each line twice, side by side, separated by one space. Every line has width `2 ** (n + 1) - 1`; strip the spaces from the right-hand end of each line.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert sierpinski(0) == [\"*\"]\n    assert sierpinski(1) == [\" *\", \"* *\"]\n    assert sierpinski(2) == [\"   *\", \"  * *\", \" *   *\", \"* * * *\"]\n    s5 = sierpinski(5)\n    assert len(s5) == 32 and s5[-1] == \" \".join(\"*\" * 32) and s5[0] == \" \" * 31 + \"*\"")

## Warm-up

It prints 1, 2, 3, counting **up**, even though `n` starts at 3 and goes down. The `print` comes **after** the recursive call, so it only happens on the way back up. That's what today is about.

## Frames and the stack

Every time any function is called, recursive or not, Python creates a **frame**: a small block of memory holding

- the function's **local variables**, including its parameters;
- **where to carry on** in the caller when this call returns.

Frames live on the **call stack**. A call **pushes** a new frame on top; a return **pops** it off, hands the return value to the frame underneath, and that frame carries on from where it stopped. Only the frame on top is running; every frame below it is waiting.

When `area` is running, the stack holds `house_cost`'s frame, then `room_cost`'s, then `area`'s on top. A traceback is a printout of exactly this stack at the moment of the error, oldest call first. Recursion is nothing special: the frames just happen to belong to the same function, each with its own copy of the parameters.

## On the way down, on the way up

Anything a recursive function does **before** its recursive call happens on the way down, as the frames pile up. Anything it does **after** happens on the way back up, as they come off, in the reverse order:

So the same recursion can count in either direction, depending on where the work goes. Step through it and watch the stack:

Reversing a string is a neat use of the way back up. The leap of faith says: reverse the rest, then put the first character at the end.

## Trees of calls

When a function calls itself **more than once**, the calls form a tree. `fib(n)` calls `fib(n - 1)` and `fib(n - 2)`, and each of them calls two more:

The stack never holds more than about n frames at once, because the tree is only n levels deep. But the **number** of calls is the size of the whole tree, and it grows explosively:

Each extra 5 in n multiplies the work by about 11. `fib(40)` takes over 300 million calls, and `fib(100)` would take longer than the age of the universe. Yet the same values are worked out over and over: `fib(25)` works out `fib(2)` 46,368 times. Next week you'll make it instant by remembering answers already found.

(The `global calls` line lets the function change a variable outside itself. It's handy for measuring like this, but don't use it for real work: a function that changes outside state is harder to reason about.)

## A shared list

A recursive function that builds a list can go wrong in a way that's hard to see. A default value is made **once**, when the function is defined, so every call that uses the default shares the same list (Week 7's trap):

The second call returns `[4, 5, 1, 2, 3]`: the list from the first call was still there. The usual fixes are to **return** new lists and combine them (no shared list at all), or to use `None` as the default and make a new list in the first call:

## Practice

## Quiz

## Daily challenge

## Before you go

Draw the tree of calls for `fib(4)` on paper, numbering the calls in the order they're made. Then mark, for each moment, which frames are on the stack. It's worth doing once by hand. Tomorrow: recursion on strings and lists, where the leap of faith really pays off.